# Quickstart — compose assumptions and bound an effect

**partialid** returns numeric partial-identification bounds for discrete causal
models. The main idea: pick literature assumptions, compose them with `&`, and
call `bound`.


In [ ]:
from partialid import Assumptions, Query, Variables, bound, from_counts

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
print(Q.describe())


## Manski bounds (order-only / maximal confounding)


In [ ]:
r = bound(V, P, Q, Assumptions.manski("X", "Y"))
print(r.summary(Q))
print("JSON-friendly:", r.to_dict()["interval"])


## Compose literature atoms with `&`

Adding assumptions can only shrink (or leave unchanged) the identified interval.


In [ ]:
base = Assumptions.order("X", "Y")
for label, a in [
    ("order only", base),
    ("+ MTR", base & Assumptions.mtr("Y", "X")),
    ("+ MTS", base & Assumptions.mts("X", "Y")),
    ("+ MTR + Tan MSM Λ=2", base & Assumptions.mtr("Y", "X") & Assumptions.tan_msm("X", 2.0)),
]:
    rr = bound(V, P, Q, a)
    print(f"{label:22s}  [{rr.lower:.4f}, {rr.upper:.4f}]  width={rr.width:.4f}  method={rr.method}")

print("\nAssumption set:")
for line in (base & Assumptions.mtr("Y", "X") & Assumptions.tan_msm("X", 2.0)).describe():
    print(" ", line)


## From counts (normalized contingency table)


In [ ]:
data = from_counts(
    ("X", "Y"),
    {(0, 0): 30, (0, 1): 20, (1, 0): 15, (1, 1): 35},
)
r_counts = bound(data.variables, data.P, Q, Assumptions.manski("X", "Y"))
print(r_counts.summary(Q))
assert abs(r_counts.width - r.width) < 1e-9


## ATE under the same table


In [ ]:
Q_ate = Query.ate(outcome="Y", treatment="X")
r_ate = bound(V, P, Q_ate, Assumptions.manski("X", "Y"))
print(r_ate.summary(Q_ate))
